### vLLM for efficient serving

In [1]:
### Init...
### df load
### model call vllm
### get df response
### close vllm server
### clean df response
### call siglip
### get score
### close siglip server 

In [2]:
from vllm import LLM, SamplingParams
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
import torch
import gc
import time
sys.path.append('./utils')

INFO 04-12 16:09:07 [__init__.py:239] Automatically detected platform cuda.


### Define

In [3]:
model_path="./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1"
global model_path

In [4]:
from vllm_model_server_start import vllm_model_server_start
from start_siglip_server import start_siglip_server
from siglip_class import SVGMetricEvaluator
from terminate_server import terminate_server

### Start vLLM model server

In [5]:
vllm_process= vllm_model_server_start(model_path)

vLLM server started with PID: 71071
Waiting for vLLM model to load...
INFO 04-12 16:09:11 [__init__.py:239] Automatically detected platform cuda.
INFO 04-12 16:09:11 [api_server.py:981] vLLM API server version 0.8.2
INFO 04-12 16:09:11 [api_server.py:982] args: Namespace(subparser='serve', model_tag='./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1', config='', host='127.0.0.1', port=8000, uvicorn_log_level='info', disable_uvicorn_access_log=False, allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key='my-api-key', lora_modules=None, prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, enable_ssl_refresh=False, ssl_cert_reqs=0, root_path=None, middleware=[], return_tokens_as_token_ids=False, disable_frontend_multiprocessing=False, enable_request_id_headers=False, enable_auto_tool_choice=False, tool_call_parser=None, tool_parser_plug

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:00<00:00,  1.23it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.93it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.78it/s]



INFO 04-12 16:09:24 [loader.py:447] Loading weights took 1.15 seconds
INFO 04-12 16:09:24 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 1.282085 seconds
INFO 04-12 16:09:30 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/18d80ded3c/rank_0_0 for vLLM's torch.compile
INFO 04-12 16:09:30 [backends.py:425] Dynamo bytecode transform time: 5.86 s
INFO 04-12 16:09:30 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-12 16:09:35 [monitor.py:33] torch.compile takes 5.86 s in total
INFO 04-12 16:09:35 [kv_cache_utils.py:566] GPU KV cache size: 25,296 tokens
INFO 04-12 16:09:35 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 24.70x
INFO 04-12 16:09:49 [gpu_model_runner.py:1534] Graph capturing finished in 14 secs, took 0.42 GiB
INFO 04-12 16:09:49 [core.py:151] init engine (profile, create kv cache, warmup model) took 25.22 seconds
WARNING 04-12 16:09:49 [config.py:1028] Default samp

INFO:     Started server process [71071]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


INFO 04-12 16:09:59 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO:     127.0.0.1:60082 - "GET /health HTTP/1.1" 200 OK
vLLM server is ready.


### OpenAI style client

In [16]:
from openai import OpenAI
# Set OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "my-api-key"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

def get_reponse(description):
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
        Write a response that appropriately completes the request.

                ### Instruction:
                Please write a SVG code for the given input.

                ### Input:
                {}

                ### Response:
                """
    
    formatted_input = alpaca_prompt.format(description)
    chat_response = client.chat.completions.create(
        model=model_path,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"{formatted_input}"},
            ],
        seed=123
    )
    return chat_response.choices[0].message.content


### Load df

In [18]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test.csv',header=[0])
print(df.shape)

(76, 3)


### Concurrent calls to vLLM server 

In [7]:
import time
from tqdm import tqdm
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# Wrap tqdm over futures
def parallel_apply_with_tqdm(func, data, max_workers=8):
    results = [None] * len(data)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(func, data[i]): i for i in range(len(data))}
        for future in tqdm(as_completed(futures), total=len(data)):
            idx = futures[future]
            try:
                results[idx] = future.result()
            except Exception as e:
                results[idx] = None
                print(f"Error at index {idx}: {e}")
    return results

# Example usage
start_time = time.time()
df['response'] = parallel_apply_with_tqdm(get_reponse, df['description'].tolist(), max_workers=12)

end_time = time.time()
print(f"Total time taken: {end_time - start_time:.2f} seconds")


  0%|                                                    | 0/76 [00:00<?, ?it/s]

INFO 04-12 16:10:08 [chat_utils.py:379] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
INFO 04-12 16:10:08 [logger.py:39] Received request chatcmpl-814f917a19fd43f6afda42c4332a8f1a: prompt: "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 12 Apr 2025\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBelow is an instruction that describes a task, paired with an input that provides further context. \n        Write a response that appropriately completes the request.\n\n                ### Instruction:\n                Please write a SVG code for the given input.\n\n                ### Input:\n                 'Morning dew on grass',\n\n                ### Response:<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=

  1%|▌                                           | 1/76 [00:03<03:47,  3.03s/it]

INFO:     127.0.0.1:60130 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO 04-12 16:10:11 [logger.py:39] Received request chatcmpl-afed62fdd7824f53b8d354ab47f888be: prompt: "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 12 Apr 2025\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBelow is an instruction that describes a task, paired with an input that provides further context. \n        Write a response that appropriately completes the request.\n\n                ### Instruction:\n                Please write a SVG code for the given input.\n\n                ### Input:\n                 'Snow-capped mountains',\n\n                ### Response:<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.6, top_p=0.9, top_k=-1, min_p=0.0, seed=None, stop=[], stop_token_ids

  4%|█▋                                          | 3/76 [00:04<01:13,  1.01s/it]

...

INFO:     127.0.0.1:60118 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:60134 - "POST /v1/chat/completions HTTP/1.1" 200 OK


 99%|██████████████████████████████████████████▍| 75/76 [00:38<00:00,  3.05it/s]

INFO:     127.0.0.1:60122 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:60094 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:60110 - "POST /v1/chat/completions HTTP/1.1" 200 OK


100%|███████████████████████████████████████████| 76/76 [00:38<00:00,  1.96it/s]

INFO:     127.0.0.1:60132 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Total time taken: 38.76 seconds


### Terminate vLLM Server

In [8]:
terminate_server(vllm_process)

INFO 04-12 16:10:47 [launcher.py:74] Shutting down FastAPI HTTP server.


[rank0]:[W412 16:10:47.147878417 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


vLLM running: False, GPU used: 747.31 MB, Below threshold: True


{'vllm_running': False,
 'gpu_memory_used_mb': 747.307008,
 'below_threshold': True}

### Start sl server

In [9]:
server_process=start_siglip_server()

server starting...
Server running: True


### Get sl score from sl server

In [11]:
import pandas as pd
import httpx

API_URL = "http://127.0.0.1:8000/evaluate_svg"
API_KEY = "my-api-key"

def send_request(client, prompt, svg):
    try:
        response = client.post(
            API_URL,
            headers={"x-api-key": API_KEY},
            json={"prompt": prompt, "svg": svg},
            timeout=10.0
        )
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e)}

def evaluate_all(df):
    with httpx.Client() as client:
        results = []
        for _, row in df.iterrows():
            result = send_request(client, row["description"], row["gpt_svg"])
            results.append(result)
        return results


# Run evaluation
results = evaluate_all(df)
# Append score or error
df["score"] = [r.get("score") if "score" in r else r.get("error") for r in results]


In [12]:
print(df['score'].mean())

0.9176208894503745

### Terminate sl server

In [13]:
terminate_server(server_process)

vLLM running: False, GPU used: 751.70 MB, Below threshold: True


{'vllm_running': False,
 'gpu_memory_used_mb': 751.69792,
 'below_threshold': True}